# RAG Implementation with LangChain and Google Gemini

This notebook demonstrates how to build a Retrieval Augmented Generation (RAG) system using LangChain, ChromaDB, and Google's Gemini models. The system will load text data, split it into chunks, create embeddings, store them in a vector database, and then use a Large Language Model (LLM) to answer questions based on the retrieved context.

In [ ]:
!pip install -qU langchain-google-genai langchain-community langchain chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 k

## 1. Setup and Installation

First, we install the necessary libraries for LangChain, Google Gemini integration, and ChromaDB.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


/tmp/ipykernel_3810/2034137088.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 2. Import Libraries

We import all the required modules from LangChain, including document loaders, text splitters, embedding models, chat models, vector stores, prompt templates, output parsers, and runnables.

In [ ]:
import os
import getpass
import sys

## 3. Google API Key Setup

To interact with Google Gemini models, we need to set up the Google API Key. It's good practice to obtain your API key from [Google AI Studio](https://aistudio.google.com/app/apikey) and store it securely, for instance, using Colab's secrets manager. For this demonstration, we're setting it directly from a variable.

In [ ]:
GOOGLE_API_KEY = "abnvcsdbvkgsddfsdhksvgkhljljsdjv"
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

In [ ]:
def setup_env():
  if os.getenv('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter your google API key")

## 4. Data Loading and Chunking

We define a function to load our text document and split it into smaller, manageable chunks. This is crucial for RAG, as LLMs have token limits, and smaller chunks allow for more precise retrieval.

In [ ]:
def load_and_split(filepath):
  if not os.path.exists(filepath):
    print(f'Error:File not found at {filepath}')
    sys.exit(1)

  print(f'Loading Data')
  loader = TextLoader(filepath)
# For pdf -> loader = PyPDFLoader(filepath)
  docs = loader.load()
  # read file -> converts into documnets

  print(f'splitting Data')
  splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=200)
# max character = 500 -> chunk_size
# overlap = 200 -> chunk_overlap
  splits = splitter.split_documents(docs) # Final chunks ready
  print(f'split {len(splits)} chunks')
  return splits


## 5. Create the RAG Chain

This section defines the core RAG chain. It involves several components:

*   **Embeddings:** Converting text chunks into numerical vectors using `gemini-embedding-001`.
*   **Vector Store:** Storing and indexing these embeddings for efficient retrieval (using ChromaDB).
*   **Retriever:** Fetching relevant document chunks based on a query.
*   **Large Language Model (LLM):** Generating the answer based on the retrieved context and the user's question (using `gemini-pro`).
*   **Prompt Template:** Structuring the input to the LLM.
*   **Output Parser:** Extracting the final answer from the LLM's response.

In [ ]:
def create_rag_chain(splits):
  embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001",task_type='retrieval_document')

  # Store in Vector DB
  vectorstore = Chroma.from_documents(documents=splits,embedding=embeddings)
  # retriever -> vector serach (search engine)
  retriever = vectorstore.as_retriever()

  llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0)
# temperature = 0 -> fixed /determinstic answer -> no randomess

  # Create a context/Prompt Template
  template = """ Answer the question based only on the following context: {context}
  Question: {question}"""

  prompt = ChatPromptTemplate.from_template(template) # converting string -> usable prompt
  def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

  chain = (
        {"context":retriever | format_docs,'question': RunnablePassthrough()}
             | prompt
             |llm
             |StrOutputParser())
  return chain

## 6. Execution

Finally, we load our specific document, create the RAG chain, and then invoke it with a user query.

In [ ]:
file_path ="/content/harrypotter.txt"
splits = load_and_split(file_path)
chain = create_rag_chain(splits)

Loading Data
splitting Data
split 15 chunks


In [ ]:
message = input("Enter your question")
output = chain.invoke(message)
print(output)

Enter your questionhow is harry


Based on the context provided, Harry is:

*   **"The Boy Who Lived"**
*   A **wizard** who enrolls at Hogwarts at age eleven.
*   Sorted into **Gryffindor House**.
*   **Befriends Ron Weasley and Hermione Granger**.
*   **Brave, impulsive, and emotionally loyal**.
*   He **confronts Voldemort** and prevents his return (in Book 1).
*   He **discovers the monster in the Chamber of Secrets is a Basilisk** and that Voldemort's teenage memory manipulated events.
*   He is **being prepared by Dumbledore for Voldemort's defeat**.
*   He **learns Voldemort's origin** and the concept of Horcruxes.
*   He is destined to fight Voldemort, as **Voldemort must kill Harry or vice versa**.
